In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
df=pd.read_csv('Churn_Modelling.csv')

In [4]:
df.drop(columns=['RowNumber', 'CustomerId','Surname'], inplace=True)

In [5]:
import pickle

In [6]:
df.Gender=df['Gender'].map({'Female':0,'Male':1})

In [7]:
from sklearn.preprocessing import OneHotEncoder
import pickle

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

encoded = encoder.fit_transform(df[['Geography']])

with open('encoder.pkl', 'wb') as f:
    pickle.dump(encoder, f)

In [9]:
df = pd.get_dummies(df, columns=['Geography'], dtype=int)

In [10]:
with open("feature_columns.pkl", "wb") as file:
    pickle.dump(df.columns.tolist(), file)

In [11]:
df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1,0,0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0,0,1
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1,0,0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1,0,0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,96270.64,0,1,0,0
9996,516,1,35,10,57369.61,1,1,1,101699.77,0,1,0,0
9997,709,0,36,7,0.00,1,0,1,42085.58,1,1,0,0
9998,772,1,42,3,75075.31,2,1,0,92888.52,1,0,1,0


In [12]:
X = df.drop('EstimatedSalary', axis=1)
y = df['EstimatedSalary']

In [55]:
X

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,1,1,0,0
1,608,0,41,1,83807.86,1,0,1,0,0,0,1
2,502,0,42,8,159660.80,3,1,0,1,1,0,0
3,699,0,39,1,0.00,2,0,0,0,1,0,0
4,850,0,43,2,125510.82,1,1,1,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,0,1,0,0
9996,516,1,35,10,57369.61,1,1,1,0,1,0,0
9997,709,0,36,7,0.00,1,0,1,1,1,0,0
9998,772,1,42,3,75075.31,2,1,0,1,0,1,0


In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

In [16]:
from sklearn.preprocessing import StandardScaler

X_scaler = StandardScaler()

X_train = X_scaler.fit_transform(X_train)
X_test = X_scaler.transform(X_test)

In [17]:
from sklearn.preprocessing import StandardScaler

y_scaler = StandardScaler()

y_train = y_scaler.fit_transform(y_train.values.reshape(-1,1))
y_test = y_scaler.transform(y_test.values.reshape(-1,1))

In [18]:
with open("X_scaler.pkl", "wb") as file:
    pickle.dump(X_scaler, file)
with open("y_scaler.pkl", "wb") as file:
    pickle.dump(y_scaler, file)

In [19]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

In [23]:
## build ann model
model=Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)), #hl1
    Dense(32, activation='relu'),#hl2
    Dense(16, activation='relu'),#hl3
    Dense(8, activation='relu'),#hl4
    Dense(4, activation='relu'),#hl5
    Dense(1)#output layer
])

In [22]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 4)              │            36 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │             5 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,617 (14.13 KB)

 Trainable params: 3,617 (14.13 KB)

 Non-trainable params: 0 (0.00 B)

In [25]:
opt=tf.keras.optimizers.Adam(learning_rate=0.001)


In [44]:
model.compile(optimizer=opt, loss='mean_squared_error', metrics=['root_mean_squared_error'])

In [45]:
## set up tensorboard
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

In [46]:
tensorflow_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [47]:
# setup early stopping
early_stopping_callback = EarlyStopping(monitor='val_loss', patience=100, restore_best_weights=True)
tensorflow_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [48]:
# train the model
history =model.fit(
    X_train,
    y_train,validation_data=(X_test, y_test),
    epochs=100,
    callbacks=[early_stopping_callback, tensorflow_callback]
)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.9957 - root_mean_squared_error: 0.9978 - val_loss: 1.0040 - val_root_mean_squared_error: 1.0020
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.9944 - root_mean_squared_error: 0.9972 - val_loss: 1.0056 - val_root_mean_squared_error: 1.0028
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.9930 - root_mean_squared_error: 0.9965 - val_loss: 1.0051 - val_root_mean_squared_error: 1.0025
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.9920 - root_mean_squared_error: 0.9960 - val_loss: 1.0086 - val_root_mean_squared_error: 1.0043
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.9902 - root_mean_squared_error: 0.9951 - val_loss: 1.0079 - val_root_mean_squared_error: 1.0039
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.9891 - root_mean_squared_error: 0.9945 - val_loss: 1.0112 - val_root_mean_squared_error: 1.0056
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2m

In [51]:
df['EstimatedSalary'].describe()

count     10000.000000
mean     100090.239881
std       57510.492818
min          11.580000
25%       51002.110000
50%      100193.915000
75%      149388.247500
max      199992.480000
Name: EstimatedSalary, dtype: float64

In [52]:
model.save('model.h5')

In [54]:
df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1,0,0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0,0,1
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1,0,0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1,0,0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,96270.64,0,1,0,0
9996,516,1,35,10,57369.61,1,1,1,101699.77,0,1,0,0
9997,709,0,36,7,0.00,1,0,1,42085.58,1,1,0,0
9998,772,1,42,3,75075.31,2,1,0,92888.52,1,0,1,0
